In [1]:
import sys
import os 
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../"))
sys.path.append(os.path.abspath("../3_Helpers"))
from optimizer import variables

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
optimization_output = pd.read_csv("Optimization_data/optimization_results_bess_pv_price_100.csv").iloc[:,:4]

In [3]:
optimization_output[:10]

,price_col,gen_scenario,bess_duration_h,sum_actual_revenue_row_total
0,Year 1,0,2.0,9.202863e+05
1,Year 1,0,2.5,1.038572e+06
2,Year 1,0,3.0,1.106216e+06
3,Year 1,0,3.5,1.165870e+06
4,Year 1,0,4.0,1.204741e+06
5,Year 1,0,4.5,1.228436e+06
6,Year 1,0,5.0,1.242489e+06
7,Year 1,0,5.5,1.253506e+06
8,Year 1,0,6.0,1.262667e+06
9,Year 1,0,6.5,1.269694e+06


In [4]:
optimization_output[optimization_output['gen_scenario'] == 7].groupby('bess_duration_h')['sum_actual_revenue_row_total'].quantile([0.05, 0.50, 0.95]).unstack()

,0.05,0.50,0.95
bess_duration_h,,,
2.0,1.019791e+06,1.379038e+06,1.875686e+06
2.5,1.052930e+06,1.421424e+06,1.946639e+06
3.0,1.067459e+06,1.452260e+06,2.009041e+06
3.5,1.086897e+06,1.478795e+06,2.067018e+06
4.0,1.103518e+06,1.505630e+06,2.118243e+06
4.5,1.118117e+06,1.523657e+06,2.159512e+06
5.0,1.130470e+06,1.543835e+06,2.180519e+06
5.5,1.140716e+06,1.557219e+06,2.198193e+06
6.0,1.149530e+06,1.568065e+06,2.212904e+06


In [5]:
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots

quantiles = {'p5': (0.05, 'dot'), 'p50': (0.50, 'dash'), 'p95': (0.95, 'dashdot')}

durations = sorted(optimization_output['bess_duration_h'].unique())
gen_scenarios = sorted(optimization_output['gen_scenario'].unique())
d_min, d_max = min(durations), max(durations)

n_cols = 3
n_rows = -(-len(gen_scenarios) // n_cols)  # ceiling division

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'PV scenario: {g}' for g in gen_scenarios],
    shared_xaxes=False, shared_yaxes=False,
)

def duration_color(d):
    t = (d - d_min) / (d_max - d_min) if d_max != d_min else 0
    return pc.sample_colorscale('Viridis', t)[0]

for g_idx, gen in enumerate(gen_scenarios):
    row = g_idx // n_cols + 1
    col = g_idx % n_cols + 1
    gen_data = optimization_output[optimization_output['gen_scenario'] == gen]

    for duration in durations:
        color = duration_color(duration)
        data = gen_data[gen_data['bess_duration_h'] == duration]['sum_actual_revenue_row_total']
        if data.empty:
            continue

        counts, bins = np.histogram(data, bins=20, density=True)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        fig.add_trace(
            go.Scatter(
                x=bin_centers, y=counts, mode='lines',
                name=f'{duration}h',
                line=dict(color=color),
                legendgroup=f'{duration}h',
                showlegend=(g_idx == 0),
            ),
            row=row, col=col,
        )

        for label, (q, dash) in quantiles.items():
            fig.add_vline(
                x=data.quantile(q),
                line=dict(color=color, dash=dash, width=1),
                row=row, col=col,
            )

fig.update_layout(
    template='plotly_white',
    title='Revenue distribution by BESS duration — per PV scenario',
    legend_title='BESS duration',
    width=1500,
    height=400 * n_rows,
)
fig.show()


In [6]:
gen7_data = optimization_output[optimization_output['gen_scenario'] == 7]

fig = go.Figure()

for duration in durations:
    color = duration_color(duration)
    data = gen7_data[gen7_data['bess_duration_h'] == duration]['sum_actual_revenue_row_total']
    if data.empty:
        continue

    counts, bins = np.histogram(data, bins=20, density=True)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    fig.add_trace(go.Scatter(x=bin_centers, y=counts, mode='lines', name=f'{duration}h', line=dict(color=color)))

    for label, (q, dash) in quantiles.items():
        fig.add_vline(
            x=data.quantile(q),
            line=dict(color=color, dash=dash, width=1),
            annotation_text=label if duration == durations[0] else '',
            annotation_position='top',
        )

fig.update_layout(
    template='plotly_white',
    title='Revenue distribution by BESS duration — PV scenario 7',
    xaxis_title='Annual revenue ($)',
    yaxis_title='Probability density',
    legend_title='BESS duration',
    width=1500,
    height=600,
)
fig.show()


In [7]:
from IPython.display import display

fmt = {
    'bess_duration_h': '{:.1f}h',
    'p5': '${:,.0f}',
    'p50': '${:,.0f}',
    'p95': '${:,.0f}',
    'p5_rank': '{:,.0f}',
    'p50_rank': '{:,.0f}',
    'p95_rank': '{:,.0f}',
}

for gen in sorted(optimization_output['gen_scenario'].unique()):
    gen_data = optimization_output[optimization_output['gen_scenario'] == gen]

    duration_quantiles = (
        gen_data
        .groupby('bess_duration_h')['sum_actual_revenue_row_total']
        .quantile([0.05, 0.5, 0.95])
        .unstack()
        .rename(columns={0.05: 'p5', 0.5: 'p50', 0.95: 'p95'})
        .reset_index()
    )

    for col in ['p5', 'p50', 'p95']:
        duration_quantiles[f'{col}_rank'] = duration_quantiles[col].rank(ascending=False).astype(int)

    duration_quantiles['p5_p50_ratio'] = duration_quantiles['p5'] / duration_quantiles['p50']

    display(
        duration_quantiles.sort_values('p50_rank').style
        .format({**fmt, 'p5_p50_ratio': '{:.2%}'})
        .set_caption(f'BESS duration ranking — PV scenario: {gen}')
    )


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$261,041","$574,520","$1,209,814",1,1,1,45.44%
11,7.5h,"$259,300","$569,162","$1,204,888",2,2,2,45.56%
10,7.0h,"$257,436","$563,620","$1,199,702",3,3,3,45.68%
9,6.5h,"$255,394","$556,938","$1,192,194",4,4,4,45.86%
8,6.0h,"$253,110","$550,427","$1,183,930",5,5,5,45.98%
7,5.5h,"$250,455","$544,144","$1,174,226",6,6,6,46.03%
6,5.0h,"$247,365","$535,062","$1,162,838",7,7,7,46.23%
5,4.5h,"$243,922","$520,863","$1,147,304",8,8,8,46.83%
4,4.0h,"$234,076","$502,600","$1,093,839",9,9,9,46.57%
3,3.5h,"$225,101","$484,191","$1,034,841",10,10,10,46.49%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$401,435","$730,573","$1,398,038",1,1,1,54.95%
11,7.5h,"$399,140","$726,252","$1,393,865",2,2,2,54.96%
10,7.0h,"$396,696","$721,517","$1,389,181",3,3,3,54.98%
9,6.5h,"$394,081","$716,207","$1,382,700",4,4,4,55.02%
8,6.0h,"$391,284","$710,096","$1,375,353",5,5,5,55.10%
7,5.5h,"$388,274","$702,937","$1,365,646",6,6,6,55.24%
6,5.0h,"$383,195","$694,286","$1,349,839",7,7,7,55.19%
5,4.5h,"$379,032","$683,581","$1,325,126",8,8,8,55.45%
4,4.0h,"$373,388","$671,502","$1,273,023",9,9,9,55.60%
3,3.5h,"$367,025","$652,916","$1,214,344",10,10,10,56.21%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$537,605","$887,566","$1,550,747",1,1,1,60.57%
11,7.5h,"$535,904","$883,741","$1,546,582",2,2,2,60.64%
10,7.0h,"$534,100","$879,551","$1,542,102",3,3,3,60.72%
9,6.5h,"$532,111","$874,646","$1,537,221",4,4,4,60.84%
8,6.0h,"$529,734","$869,490","$1,531,343",5,5,5,60.92%
7,5.5h,"$526,758","$861,571","$1,523,593",6,6,6,61.14%
6,5.0h,"$522,716","$849,975","$1,505,771",7,7,7,61.50%
5,4.5h,"$517,658","$836,301","$1,481,712",8,8,8,61.90%
4,4.0h,"$508,494","$818,231","$1,437,669",9,9,9,62.15%
3,3.5h,"$498,209","$799,004","$1,384,124",10,10,10,62.35%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$659,608","$1,046,812","$1,681,133",1,1,1,63.01%
11,7.5h,"$657,715","$1,041,951","$1,677,595",2,2,2,63.12%
10,7.0h,"$655,443","$1,036,295","$1,673,617",3,3,3,63.25%
9,6.5h,"$652,341","$1,029,927","$1,669,275",4,4,4,63.34%
8,6.0h,"$647,912","$1,024,675","$1,663,966",5,5,5,63.23%
7,5.5h,"$644,233","$1,015,113","$1,656,210",6,6,6,63.46%
6,5.0h,"$642,276","$1,004,458","$1,642,619",7,7,7,63.94%
5,4.5h,"$635,703","$991,170","$1,620,166",8,8,8,64.14%
4,4.0h,"$624,027","$973,345","$1,582,398",9,9,9,64.11%
3,3.5h,"$610,858","$952,845","$1,538,510",10,10,10,64.11%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$771,821","$1,193,841","$1,830,712",1,1,1,64.65%
11,7.5h,"$769,476","$1,189,767","$1,823,156",2,2,2,64.67%
10,7.0h,"$766,494","$1,184,897","$1,813,877",3,3,3,64.69%
9,6.5h,"$762,392","$1,178,828","$1,802,468",4,4,4,64.67%
8,6.0h,"$760,680","$1,168,091","$1,789,188",5,5,5,65.12%
7,5.5h,"$759,123","$1,159,066","$1,781,702",6,6,6,65.49%
6,5.0h,"$757,306","$1,148,593","$1,771,650",7,7,7,65.93%
5,4.5h,"$755,248","$1,132,297","$1,749,732",8,8,8,66.70%
4,4.0h,"$749,396","$1,113,942","$1,713,986",9,9,9,67.27%
3,3.5h,"$736,765","$1,092,144","$1,672,833",10,10,10,67.46%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$908,570","$1,331,477","$1,990,699",1,1,1,68.24%
11,7.5h,"$906,623","$1,327,209","$1,986,351",2,2,2,68.31%
10,7.0h,"$904,430","$1,322,354","$1,981,025",3,3,3,68.40%
9,6.5h,"$901,928","$1,316,599","$1,973,701",4,4,4,68.50%
8,6.0h,"$899,069","$1,309,025","$1,964,181",5,5,5,68.68%
7,5.5h,"$895,805","$1,297,321","$1,951,670",6,6,6,69.05%
6,5.0h,"$891,949","$1,284,188","$1,935,548",7,7,7,69.46%
5,4.5h,"$884,400","$1,268,854","$1,908,551",8,8,8,69.70%
4,4.0h,"$872,174","$1,246,708","$1,870,628",9,9,9,69.96%
3,3.5h,"$861,935","$1,225,228","$1,822,263",10,10,10,70.35%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$1,045,399","$1,473,392","$2,121,945",1,1,1,70.95%
11,7.5h,"$1,043,393","$1,468,817","$2,117,025",2,2,2,71.04%
10,7.0h,"$1,041,102","$1,463,120","$2,110,693",3,3,3,71.16%
9,6.5h,"$1,038,557","$1,454,836","$2,102,469",4,4,4,71.39%
8,6.0h,"$1,033,293","$1,444,712","$2,091,812",5,5,5,71.52%
7,5.5h,"$1,025,211","$1,432,913","$2,078,404",6,6,6,71.55%
6,5.0h,"$1,015,602","$1,413,990","$2,062,037",7,7,7,71.83%
5,4.5h,"$1,004,462","$1,397,925","$2,040,068",8,8,8,71.85%
4,4.0h,"$990,850","$1,379,608","$1,998,308",9,9,9,71.82%
3,3.5h,"$975,198","$1,355,551","$1,948,346",10,10,10,71.94%


,bess_duration_h,p5,p50,p95,p5_rank,p50_rank,p95_rank,p5_p50_ratio
12,8.0h,"$1,157,980","$1,593,603","$2,246,714",1,1,1,72.66%
11,7.5h,"$1,156,515","$1,590,285","$2,241,131",2,2,2,72.72%
10,7.0h,"$1,154,821","$1,584,229","$2,233,807",3,3,3,72.89%
9,6.5h,"$1,152,912","$1,576,831","$2,224,648",4,4,4,73.12%
8,6.0h,"$1,149,530","$1,568,065","$2,212,904",5,5,5,73.31%
7,5.5h,"$1,140,716","$1,557,219","$2,198,193",6,6,6,73.25%
6,5.0h,"$1,130,470","$1,543,835","$2,180,519",7,7,7,73.22%
5,4.5h,"$1,118,117","$1,523,657","$2,159,512",8,8,8,73.38%
4,4.0h,"$1,103,518","$1,505,630","$2,118,243",9,9,9,73.29%
3,3.5h,"$1,086,897","$1,478,795","$2,067,018",10,10,10,73.50%


In [8]:
import plotly.express as px

# --- 1. P5 / P50 / P95 vs BESS duration, one line per PV scenario ---
quantile_by_gen_duration = (
    optimization_output
    .groupby(['gen_scenario', 'bess_duration_h'])['sum_actual_revenue_row_total']
    .quantile([0.05, 0.5, 0.95])
    .unstack()
    .rename(columns={0.05: 'p5', 0.5: 'p50', 0.95: 'p95'})
    .reset_index()
)
quantile_by_gen_duration['gen_scenario'] = quantile_by_gen_duration['gen_scenario'].astype(str)


# --- 2. Marginal P5 gain per +0.5h of BESS duration ---
quantile_by_gen_duration_sorted = quantile_by_gen_duration.sort_values(['gen_scenario', 'bess_duration_h'])
quantile_by_gen_duration_sorted['marginal_p5'] = (
    quantile_by_gen_duration_sorted.groupby('gen_scenario')['p5'].diff()
)

fig2 = px.line(
    quantile_by_gen_duration_sorted.dropna(subset=['marginal_p5']),
    x='bess_duration_h', y='marginal_p5',
    color='gen_scenario', markers=True,
    title='Marginal P5 gain per +0.5h BESS duration by PV scenario',
    labels={'bess_duration_h': 'BESS duration (h)', 'marginal_p5': 'Marginal P5 gain ($)', 'gen_scenario': 'PV scenario'},
)
fig2.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='zero marginal gain')
fig2.update_layout(template='plotly_white', width=1400, height=500, legend_title='PV scenario')
fig2.show()

